# Ninisite Data Collection

Collects posts (opening posts and replies) from the newest topics of Ninisite forums and marks the posts that contain words from the offensive lexicon (the same JSON lexicon used for the X data).

No Ninisite account or login is needed: forums and topics are public pages. Ninisite's own search only works when logged in, so instead of searching for each word, the notebook reads the topics and finds the lexicon words in the posts itself.

Steps:
1. Check `LEXICON_FILE` and choose `FORUM_IDS` in the configuration cell. A forum id is the number in a forum link, e.g. `https://www.ninisite.com/discussion/forum/139/همسران` is forum `139`. All forums are listed on https://www.ninisite.com/discussion
2. Run the cells in order. With the default settings the collection takes about half an hour.

Each topic is saved as soon as it is collected. Topics already in `NinisitePosts.csv` are skipped on later runs, so each run adds only the topics that are new since the last run; replies added later to a saved topic are not collected.

In [ ]:
import json
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configuration

In [ ]:
LEXICON_FILE = 'Only_Persian_Offensive_lexicon.json'
OUTPUT_CSV = "NinisitePosts.csv"
FORUM_IDS = [131, 139, 108, 140, 142]  # عمومی, همسران, درد دل, بستگان دوستان, رویداد جامعه
FORUM_PAGES_PER_FORUM = 3  # pages of each forum's topic list, newest topics first, about 40 topics per page
MAX_PAGES_PER_TOPIC = 5  # pages of posts per topic, 15 posts per page
DELAY_BETWEEN_REQUESTS = 1  # seconds, to avoid being rate limited by Ninisite
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

## Read the lexicon

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({'ي': 'ی', 'ك': 'ک'})

def read_lexicon(path):
    with open(path, 'r', encoding='utf-8') as lexicon_file:
        lexicon = json.load(lexicon_file)
    if 'word' not in lexicon:
        raise ValueError('The lexicon file has no "word" list: ' + path)
    return lexicon['word']

def normalize(text):
    # Arabic yeh and kaf look the same as the Persian letters but would not match the lexicon
    return text.translate(ARABIC_TO_PERSIAN)

def make_lexicon_pattern(words):
    # Whole words only, so short words like 'کس' do not match inside longer words;
    # longest words first, so 'بی ادب' is found as one word instead of 'ادب'
    alternatives = sorted({re.escape(normalize(word.strip())) for word in words if word.strip()}, key=len, reverse=True)
    return re.compile(r'(?<!\w)(?:' + '|'.join(alternatives) + r')(?!\w)')

def find_lexicon_words(pattern, text):
    return sorted(set(pattern.findall(normalize(text))))

lexicon_words = read_lexicon(LEXICON_FILE)
lexicon_pattern = make_lexicon_pattern(lexicon_words)
len(lexicon_words)

## Get posts

In [ ]:
FORUM_URL = 'https://www.ninisite.com/discussion/forum/{forum_id}/'
TOPIC_URL = 'https://www.ninisite.com/discussion/topic/{topic_id}/'
TOPIC_LINK_PATTERN = re.compile(r'^/discussion/topic/(\d+)/[^?#]*$')
DATE_FORMAT = '%m/%d/%Y %I:%M:%S %p'

def make_session():
    # Retries dropped connections and rate limit / server errors, waiting longer after each attempt
    retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({'User-Agent': USER_AGENT})
    return session

def get_page(session, url, page):
    response = session.get(url, params={'page': page}, timeout=30)
    response.raise_for_status()
    time.sleep(DELAY_BETWEEN_REQUESTS)
    return BeautifulSoup(response.text, 'html.parser')

def get_forum_topic_ids(session, forum_id, pages):
    # Topic ids from the forum's topic list (newest first); pinned topics are on every page, so ids are de-duplicated
    topic_ids = []
    for page in range(1, pages + 1):
        soup = get_page(session, FORUM_URL.format(forum_id=forum_id), page)
        for link in soup.select('#cat a[href]'):
            match = TOPIC_LINK_PATTERN.match(link['href'])
            if match and match.group(1) not in topic_ids:
                topic_ids.append(match.group(1))
    return topic_ids

def last_page_number(soup):
    numbers = [int(link.get_text(strip=True)) for link in soup.select('.page-link') if link.get_text(strip=True).isdigit()]
    return max(numbers, default=1)

def text_or_none(tag):
    return tag.get_text(' ', strip=True) if tag else None

def post_row(forum_id, topic_id, title, article):
    # The opening post is the article with id 'topic'; replies have ids like 'post-426882572'
    is_opening_post = article.get('id') == 'topic'
    published = article.select_one('meta[itemprop=datepublished]')
    likes = article.select_one('.like-count')
    return {'forum_id': forum_id, 'topic_id': topic_id, 'topic_title': title,
            'post_id': topic_id if is_opening_post else article['id'].removeprefix('post-'),
            'source': 'topic' if is_opening_post else 'reply',
            'username': text_or_none(article.select_one('[itemprop=author] [itemprop=name]')),
            'created_at': pd.to_datetime(published['content'], format=DATE_FORMAT) if published else None,
            'likes': int(likes['data-like-count']) if likes else 0,
            'text': article.select_one('.post-message').get_text('\n', strip=True)}

def get_topic_posts(session, forum_id, topic_id, max_pages):
    url = TOPIC_URL.format(topic_id=topic_id)
    soup = get_page(session, url, 1)
    title = text_or_none(soup.select_one('h1.topic-title'))
    if title is None:
        raise ValueError('No topic in the page (deleted or not public): ' + url)
    posts = {}
    for page in range(1, min(last_page_number(soup), max_pages) + 1):
        if page > 1:
            soup = get_page(session, url, page)
        for article in soup.select('article.topic-post'):
            if article.select_one('.post-message') is None:
                continue
            row = post_row(forum_id, topic_id, title, article)
            posts[row['post_id']] = row  # the opening post is repeated on every page
    return list(posts.values())

## Save posts

In [ ]:
def clean_text(text):
    # same cleaning as the other notebooks: remove #, commas, new lines and @mentions
    text = text.replace('#', '').replace(',', ' ').replace('\n', ' ')
    return re.sub(r'@[^\s]+', '', text).strip()

def clean_rows(rows):
    cleaned = [{**row, 'text': clean_text(row['text'])} for row in rows]
    return [row for row in cleaned if row['text']]

def append_rows(path, rows, first_index):
    if not rows:
        return
    table = pd.DataFrame(rows, index=range(first_index, first_index + len(rows)))
    table.to_csv(path, mode='a', header=not os.path.exists(path), encoding='utf-8')

def mark_lexicon_words(rows):
    # badwords: lexicon words found in the post, empty when there are none
    return [{**row, 'badwords': ' | '.join(find_lexicon_words(lexicon_pattern, row['text']))} for row in rows]

def read_saved_posts(path):
    # Topics already in the output file are skipped
    if not os.path.exists(path):
        return pd.DataFrame(columns=['topic_id'])
    return pd.read_csv(path, index_col=0, dtype={'topic_id': str}, encoding='utf-8')

## Collect the topics

In [ ]:
session = make_session()
saved_posts = read_saved_posts(OUTPUT_CSV)
saved_topic_ids = set(saved_posts['topic_id'])
next_index = int(saved_posts.index.max()) + 1 if len(saved_posts) else 0
failed_topics = []
for forum_id in FORUM_IDS:
    print('=============== getting topics of forum', forum_id, '===============')
    try:
        topic_ids = get_forum_topic_ids(session, forum_id, FORUM_PAGES_PER_FORUM)
    except requests.RequestException as err:
        print('failed to get the topic list:', err)
        continue
    new_topic_ids = [topic_id for topic_id in topic_ids if topic_id not in saved_topic_ids]
    print(len(topic_ids) - len(new_topic_ids), 'topics already collected,', len(new_topic_ids), 'to collect')
    for i, topic_id in enumerate(new_topic_ids):
        try:
            topic_posts = mark_lexicon_words(clean_rows(get_topic_posts(session, forum_id, topic_id, MAX_PAGES_PER_TOPIC)))
            append_rows(OUTPUT_CSV, topic_posts, next_index)
            next_index += len(topic_posts)
            saved_topic_ids.add(topic_id)
            print(i, 'topic', topic_id, '- got', len(topic_posts), 'posts')
        except (requests.RequestException, ValueError) as err:
            print(i, 'topic', topic_id, '- failed:', err)
            failed_topics.append(topic_id)

print('failed topics:', len(failed_topics))

In [ ]:
ninisite_df = pd.read_csv(OUTPUT_CSV, index_col=0, encoding='utf-8')
bad_posts_df = ninisite_df[ninisite_df['badwords'].notna()]
print('posts:', len(ninisite_df), '| posts with lexicon words:', len(bad_posts_df))
bad_posts_df.head(900)